In [9]:
import pandas as pd
import unicodedata
from pathlib import Path
from IPython.display import display
from thefuzz import process, fuzz

print("=" * 60)
print("🛠️ INITIALIZING & AUDITING PREDICTIONS DATA")
print("=" * 60)

preds_path = Path("../data/processed/test_predictions.csv")
master_path = Path("../data/processed/player_match_stats_clean.csv")

if not preds_path.exists():
    print(f"❌ File not found at: {preds_path}")
    print("Please ensure you completed Notebook 05.")
else:
    df_preds = pd.read_csv(preds_path)
    
    # ---------------------------------------------------------
    # PART 1: The Fix (Rescuing Missing Names)
    # ---------------------------------------------------------
    if 'player_name' not in df_preds.columns:
        print("⚠️ 'player_name' missing. Rescuing from master dataset...")
        df_names = pd.read_csv(master_path, usecols=["player_id", "player_name"]).drop_duplicates(subset=["player_id"])
        df_preds = pd.merge(df_preds, df_names, on="player_id", how="left")
        
        # Reorder columns so player_name is next to player_id
        cols = df_preds.columns.tolist()
        cols.insert(1, cols.pop(cols.index('player_name')))
        df_preds = df_preds[cols]
        
        df_preds.to_csv(preds_path, index=False)
        print(f"✅ Rescued {df_preds['player_name'].notna().sum():,} names and updated file.\n")
    else:
        print("✅ 'player_name' is present. No fix needed.\n")

    # ---------------------------------------------------------
    # PART 2: The Audit
    # ---------------------------------------------------------
    total_rows = len(df_preds)
    unique_players = df_preds["player_id"].nunique()
    avg_matches = total_rows / unique_players if unique_players > 0 else 0

    print("--- Dataset Demographics ---")
    print(f"Total Rows:     {total_rows:,}")
    print(f"Unique Players: {unique_players:,}")
    print(f"Matches/Player: ~{avg_matches:.1f} (Average)\n")
    
    print("--- Columns Present ---")
    print(list(df_preds.columns))

    print("\n--- Sample Rows (First 3) ---")
    display(df_preds.head(3))
    
    print("\n" + "=" * 60)
    if total_rows > unique_players:
        print("🚨 VERDICT: Duplicates confirmed. We must aggregate by player_id.")
    print("=" * 60)

🛠️ INITIALIZING & AUDITING PREDICTIONS DATA
✅ 'player_name' is present. No fix needed.

--- Dataset Demographics ---
Total Rows:     349,947
Unique Players: 10,627
Matches/Player: ~32.9 (Average)

--- Columns Present ---
['player_id', 'player_name', 'game_id', 'date', 'position', 'market_value_in_eur', 'std_fp_last_5', 'avg_fp_last_5', 'predicted_fantasy_points', 'actual_fantasy_points', 'prediction_error']

--- Sample Rows (First 3) ---


,player_id,player_name,game_id,date,position,market_value_in_eur,std_fp_last_5,avg_fp_last_5,predicted_fantasy_points,actual_fantasy_points,prediction_error
0,434675,Cody Gakpo,4095269,2024-01-01,0,70000000.0,2.509980,3.4,3.824497,6,-2.175503
1,451276,Dominik Szoboszlai,4095269,2024-01-01,3,85000000.0,2.387467,2.8,3.221833,2,1.221833
2,478573,Ryan Gravenberch,4095269,2024-01-01,3,90000000.0,0.547723,1.4,2.132562,1,1.132562



🚨 VERDICT: Duplicates confirmed. We must aggregate by player_id.


In [10]:
# ---------------------------------------------------------
# PART 2: The Duplicate Audit
# ---------------------------------------------------------
total_rows = len(df_preds)
unique_players = df_preds["player_id"].nunique()
duplicate_count = total_rows - unique_players

print("=" * 60)
print("📊 PLAYER DUPLICATION AUDIT")
print("=" * 60)
print(f"Total Rows in Test Set (len):  {total_rows:,}")
print(f"Unique Players (.nunique()):    {unique_players:,}")
print(f"Total Duplicate Rows:           {duplicate_count:,}")

if total_rows > unique_players:
    duplication_rate = (duplicate_count / total_rows) * 100
    print(f"\n🚨 CONFIRMED: {duplication_rate:.1f}% of the rows contain repeating players.")
    print("This happens because a player has a separate row for every match they played in 2024+.")
    print("We must group/aggregate by player_id before sorting to build unique player rankings.")
else:
    print("\n✅ Clean 1:1 mapping found (No duplicates).")
print("=" * 60)

📊 PLAYER DUPLICATION AUDIT
Total Rows in Test Set (len):  349,947
Unique Players (.nunique()):    10,627
Total Duplicate Rows:           339,320

🚨 CONFIRMED: 97.0% of the rows contain repeating players.
This happens because a player has a separate row for every match they played in 2024+.
We must group/aggregate by player_id before sorting to build unique player rankings.


In [12]:
print("=" * 60)
print("📸 CREATING LATEST PLAYER SNAPSHOT")
print("=" * 60)

# 1. Ensure 'date' is a datetime object for accurate chronological sorting
df_preds['date'] = pd.to_datetime(df_preds['date'])

# 2. Sort by player and date (newest first)
df_preds_sorted = df_preds.sort_values(by=['player_id', 'date'], ascending=[True, False])

# 3. Drop duplicates to keep only the first (latest) row per player
df_latest_snapshot = df_preds_sorted.drop_duplicates(subset=['player_id'], keep='first').copy()

# 4. Audit the Snapshot
total_rows = len(df_latest_snapshot)
unique_players = df_latest_snapshot['player_id'].nunique()

print("--- Snapshot Demographics ---")
print(f"Rows (Snapshots): {total_rows:,}")
print(f"Unique Players:   {unique_players:,}")

if total_rows == unique_players:
    print("✅ VERDICT: 1:1 Mapping Achieved. No duplicate players in snapshot.\n")
else:
    print("🚨 VERDICT: Duplicates still exist. Check sorting and drop logic.\n")

print("--- Sample Records (First 5) ---")
display(df_latest_snapshot.head(5))

📸 CREATING LATEST PLAYER SNAPSHOT
--- Snapshot Demographics ---
Rows (Snapshots): 10,627
Unique Players:   10,627
✅ VERDICT: 1:1 Mapping Achieved. No duplicate players in snapshot.

--- Sample Records (First 5) ---


,player_id,player_name,game_id,date,position,market_value_in_eur,std_fp_last_5,avg_fp_last_5,predicted_fantasy_points,actual_fantasy_points,prediction_error
349929,3333,James Milner,4626168,2026-05-24,3,750000.0,0.000000,2.0,1.998346,1,0.998346
63015,4391,Boy Waterman,4098161,2024-05-19,2,50000.0,3.130495,1.6,2.375754,1,1.375754
325897,5336,Anastasios Tsokanis,4665115,2026-03-22,3,200000.0,0.447214,1.2,1.684353,1,0.684353
346512,7161,Jonas Hofmann,4634550,2026-05-16,3,2000000.0,0.447214,1.2,1.707809,1,0.707809
204637,7825,Pepe Reina,4374294,2025-05-23,2,600000.0,3.130495,3.6,2.828659,-2,4.828659


In [14]:
print("=" * 60)
print("🔬 RECOMMENDATION ENGINE SANITY TEST")
print("=" * 60)

# 1. Sort the snapshot by predicted_fantasy_points descending
df_ranked = df_latest_snapshot.sort_values(by="predicted_fantasy_points", ascending=False).reset_index(drop=True)

# 2. Extract Top 20 Overall
top_20_players = df_ranked.head(20)[[
    "player_id", "player_name", "position", "predicted_fantasy_points", 
    "actual_fantasy_points", "market_value_in_eur", "avg_fp_last_5"
]]

# 3. Compute Position Distribution in Top 50
top_50_positions = df_ranked.head(50)["position"].value_counts().to_frame().reset_index()
top_50_positions.columns = ["Position", "Player Count"]

# --- OUTPUT DISPLAY ---
print("\n🔥 TOP 20 PLAYERS BY PREDICTED_FANTASY_POINTS")
print("-" * 45)
display(top_20_players)

print("\n📊 POSITION DISTRIBUTION IN TOP 50")
print("-" * 35)
display(top_50_positions)



🔬 RECOMMENDATION ENGINE SANITY TEST

🔥 TOP 20 PLAYERS BY PREDICTED_FANTASY_POINTS
---------------------------------------------


,player_id,player_name,position,predicted_fantasy_points,actual_fantasy_points,market_value_in_eur,avg_fp_last_5
0,418560,Erling Haaland,0,8.307174,7,200000000.0,6.000000
1,342229,Kylian Mbappé,0,6.690482,7,180000000.0,4.400000
2,411295,Raphinha,0,6.449673,12,50000000.0,8.200000
3,546543,Darwin Núñez,0,6.015812,1,70000000.0,1.000000
4,371998,Vinicius Junior,0,5.751913,7,180000000.0,5.800000
5,256448,Gianluca Mancini,1,5.257630,7,15000000.0,9.000000
6,655012,Lucas Stassin,0,5.233005,2,15000000.0,7.800000
7,146854,Wissam Ben Yedder,0,5.132897,7,8000000.0,6.400000
8,242086,Patrik Schick,0,5.001832,2,25000000.0,8.000000
9,576024,Julián Álvarez,0,4.928077,2,100000000.0,5.600000



📊 POSITION DISTRIBUTION IN TOP 50
-----------------------------------


,Position,Player Count
0,0,46
1,3,3
2,1,1


In [16]:
print("=" * 60)
print("📈 NICHE ENGINEERING: DESCRIPTIVE ANALYSIS")
print("=" * 60)

# ---------------------------------------------------------
# 1. MARKET VALUE ANALYSIS (Hidden Gems Prep)
# ---------------------------------------------------------
print("\n--- 💰 Market Value Distribution ---")
print("Comparing Full Test Data (Match-Level) vs. Latest Snapshot (Player-Level)")

# Calculate descriptive stats
mv_desc = pd.DataFrame({
    "Full Test Data": df_preds["market_value_in_eur"].describe(),
    "Latest Snapshot": df_latest_snapshot["market_value_in_eur"].describe()
})

# Format as currency for readability, removing scientific notation
# FIXED: Using .map() instead of .applymap()
formatted_mv_desc = mv_desc.map(lambda x: f"€{x:,.0f}" if pd.notnull(x) else "NaN")
display(formatted_mv_desc)

# ---------------------------------------------------------
# 2. CAPTAINCY METRICS ANALYSIS (Reliability Prep)
# ---------------------------------------------------------
print("\n--- 🛡️ Captaincy & Performance Metrics (Snapshot) ---")
print("Analyzing Predicted Points and Form Stability (std_fp_last_5)")

# Describe the prediction and standard deviation columns
cap_desc = df_latest_snapshot[["predicted_fantasy_points", "std_fp_last_5"]].describe()
display(cap_desc.round(2))

# ---------------------------------------------------------
# 3. CORRELATION CHECK
# ---------------------------------------------------------
print("\n--- 🔗 Feature Correlation (Snapshot) ---")
print("Checking baseline relationships between our target niches.")

# Check how strongly market value dictates predicted points in our snapshot
corr = df_latest_snapshot[[
    "predicted_fantasy_points", 
    "market_value_in_eur", 
    "std_fp_last_5"
]].corr()
display(corr.round(3))

print("\n" + "=" * 60)
print("🏁 READY TO DEFINE THRESHOLDS FOR GEMS & CAPTAINS")
print("=" * 60)

📈 NICHE ENGINEERING: DESCRIPTIVE ANALYSIS

--- 💰 Market Value Distribution ---
Comparing Full Test Data (Match-Level) vs. Latest Snapshot (Player-Level)


,Full Test Data,Latest Snapshot
count,"€328,703","€8,924"
mean,"€9,545,736","€4,805,357"
std,"€17,453,974","€11,329,816"
min,"€10,000","€10,000"
25%,"€800,000","€400,000"
50%,"€3,000,000","€1,000,000"
75%,"€10,000,000","€4,000,000"
max,"€200,000,000","€200,000,000"



--- 🛡️ Captaincy & Performance Metrics (Snapshot) ---
Analyzing Predicted Points and Form Stability (std_fp_last_5)


,predicted_fantasy_points,std_fp_last_5
count,10627.00,10627.00
mean,2.30,1.84
std,0.59,1.42
min,0.15,0.00
25%,1.85,0.55
50%,2.22,1.64
75%,2.66,2.83
max,8.31,9.98



--- 🔗 Feature Correlation (Snapshot) ---
Checking baseline relationships between our target niches.


,predicted_fantasy_points,market_value_in_eur,std_fp_last_5
predicted_fantasy_points,1.000,0.316,0.561
market_value_in_eur,0.316,1.000,0.096
std_fp_last_5,0.561,0.096,1.000



🏁 READY TO DEFINE THRESHOLDS FOR GEMS & CAPTAINS


### Captain Recommendations

In [17]:
print("=" * 60)
print("©️ CAPTAINCY RECOMMENDATION ENGINE")
print("=" * 60)

# Create a working copy to avoid SettingWithCopyWarning
df_captains = df_latest_snapshot.copy()

# ---------------------------------------------------------
# SCORING FORMULAS
# ---------------------------------------------------------
# 1. Best Captain (Default sorting by predicted points)

# 2. Safe Captain (Penalty for high volatility)
df_captains['safe_captain_score'] = df_captains['predicted_fantasy_points'] / (1 + df_captains['std_fp_last_5'])

# 3. Differential Captain (Multiplier for high volatility/ceiling)
df_captains['differential_captain_score'] = df_captains['predicted_fantasy_points'] * (1 + df_captains['std_fp_last_5'])


# ---------------------------------------------------------
# BUCKET EXTRACTION (Top 5 per category)
# ---------------------------------------------------------
# Columns to display for clean reading
base_cols = ["player_name", "position", "predicted_fantasy_points", "std_fp_last_5"]

# Bucket 1: Best Overall 
best_captains = df_captains.sort_values(by="predicted_fantasy_points", ascending=False).head(5)[base_cols]

# Bucket 2: Safe Picks
safe_captains = df_captains.sort_values(by="safe_captain_score", ascending=False).head(5)[base_cols + ["safe_captain_score"]]

# Bucket 3: Differential Picks
diff_captains = df_captains.sort_values(by="differential_captain_score", ascending=False).head(5)[base_cols + ["differential_captain_score"]]


# ---------------------------------------------------------
# OUTPUT DISPLAY
# ---------------------------------------------------------
print("\n👑 BUCKET 1: BEST OVERALL CAPTAINS (Max Expected Points)")
print("-" * 60)
display(best_captains)

print("\n🛡️ BUCKET 2: SAFE CAPTAINS (High Floor / Low Volatility)")
print("-" * 60)
display(safe_captains)

print("\n🚀 BUCKET 3: DIFFERENTIAL CAPTAINS (High Ceiling / Boom-or-Bust)")
print("-" * 60)
display(diff_captains)

print("\n" + "=" * 60)
print("✅ Captaincy buckets successfully calculated and sorted.")
print("=" * 60)

©️ CAPTAINCY RECOMMENDATION ENGINE

👑 BUCKET 1: BEST OVERALL CAPTAINS (Max Expected Points)
------------------------------------------------------------


,player_name,position,predicted_fantasy_points,std_fp_last_5
348657,Erling Haaland,0,8.307174,3.316625
348848,Kylian Mbappé,0,6.690482,2.509980
347057,Raphinha,0,6.449673,9.984989
205924,Darwin Núñez,0,6.015812,0.000000
346865,Vinicius Junior,0,5.751913,3.834058



🛡️ BUCKET 2: SAFE CAPTAINS (High Floor / Low Volatility)
------------------------------------------------------------


,player_name,position,predicted_fantasy_points,std_fp_last_5,safe_captain_score
205924,Darwin Núñez,0,6.015812,0.0,6.015812
349315,Iliman Ndiaye,0,3.396076,0.0,3.396076
347249,Dmitriy Pestryakov,0,3.278857,0.0,3.278857
234335,Poul Kallsberg,0,3.269849,0.0,3.269849
349912,Adam Armstrong,0,3.151755,0.0,3.151755



🚀 BUCKET 3: DIFFERENTIAL CAPTAINS (High Ceiling / Boom-or-Bust)
------------------------------------------------------------


,player_name,position,predicted_fantasy_points,std_fp_last_5,differential_captain_score
347057,Raphinha,0,6.449673,9.984989,70.849585
349898,Gianluca Mancini,1,5.257630,8.455767,49.714926
342944,Talisca,3,4.819991,7.395945,40.468379
346846,Gleiker Mendoza,0,4.104032,8.590693,39.360509
62712,Wissam Ben Yedder,0,5.132897,6.580274,38.908766



✅ Captaincy buckets successfully calculated and sorted.


### Hidden Gems (Underrated Picks)

In [18]:
print("=" * 60)
print("💎 HIDDEN GEMS RECOMMENDATION ENGINE")
print("=" * 60)

# 1. Create a working copy and drop missing market values for accurate ranking
df_gems = df_latest_snapshot.dropna(subset=['market_value_in_eur']).copy()

# 2. Calculate Percentiles (Ranked from 0 to 100)
df_gems['prediction_pct'] = df_gems['predicted_fantasy_points'].rank(pct=True) * 100
df_gems['value_pct'] = df_gems['market_value_in_eur'].rank(pct=True) * 100

# 3. Calculate Gem Score
df_gems['gem_score'] = df_gems['prediction_pct'] - df_gems['value_pct']

# 4. Apply Playability Filter: 
# We only want "Gems" who are actually good at football (Above average expected points)
df_gems_playable = df_gems[df_gems['prediction_pct'] >= 50].copy()

# 5. Extract the Top 10 Hidden Gems
top_gems = df_gems_playable.sort_values(by="gem_score", ascending=False).head(10)

# ---------------------------------------------------------
# OUTPUT FORMATTING & DISPLAY
# ---------------------------------------------------------
cols_to_show = [
    "player_name", "position", "market_value_in_eur", 
    "predicted_fantasy_points", "prediction_pct", "value_pct", "gem_score"
]
display_df = top_gems[cols_to_show].copy()

# Clean up formatting for readability
display_df['market_value_in_eur'] = display_df['market_value_in_eur'].map(lambda x: f"€{x:,.0f}")
display_df['prediction_pct'] = display_df['prediction_pct'].round(1).astype(str) + '%'
display_df['value_pct'] = display_df['value_pct'].round(1).astype(str) + '%'
display_df['gem_score'] = display_df['gem_score'].round(2)
display_df['predicted_fantasy_points'] = display_df['predicted_fantasy_points'].round(2)

print("\n🕵️ BURIED TREASURE: Top 10 Hidden Gems")
print("Filtered for players in the top 50% of expected points.")
print("-" * 60)
display(display_df)

print("\n" + "=" * 60)
print("✅ Percentile-based Gem Scores calculated successfully.")
print("=" * 60)

💎 HIDDEN GEMS RECOMMENDATION ENGINE

🕵️ BURIED TREASURE: Top 10 Hidden Gems
Filtered for players in the top 50% of expected points.
------------------------------------------------------------


,player_name,position,market_value_in_eur,predicted_fantasy_points,prediction_pct,value_pct,gem_score
348983,Carlos Rojas,0,"€100,000",3.57,97.1%,3.7%,93.39
201958,Abdoulay Diaby,0,"€175,000",3.39,95.0%,8.0%,87.06
162545,Fran Vélez,1,"€150,000",3.24,92.9%,6.5%,86.40
312353,Jota,0,"€100,000",3.11,90.0%,3.7%,86.36
327793,Chinedu Geoffrey,0,"€100,000",3.10,89.8%,3.7%,86.10
277259,Jonas Lössl,2,"€100,000",3.08,89.3%,3.7%,85.62
327059,Redouane Halhal,1,"€100,000",3.06,88.5%,3.7%,84.84
348838,Vadym Sydun,0,"€250,000",3.76,98.0%,13.3%,84.68
330891,Jordan Obita,1,"€200,000",3.34,94.5%,9.9%,84.57
313613,Daniel Høegh,1,"€150,000",3.13,90.6%,6.5%,84.03



✅ Percentile-based Gem Scores calculated successfully.


In [22]:
print("=" * 60)
print("🔗 WAREHOUSE MAPPING: SQUADS TO TRANSFERMARKT")
print("=" * 60)

# ---------------------------------------------------------
# 1. FILE PATHS & DATA LOADING
# ---------------------------------------------------------
squad_path = Path("../scrapers/fifa_squads/output/wc2026_players.json") 
master_path = Path("../data/processed/player_match_stats_clean.csv")

df_squads = pd.read_json(squad_path)
df_squads['wc_player_id'] = df_squads.index 

# Load Master Transfermarkt database (Unique players only)
df_master = pd.read_csv(master_path, usecols=["player_id", "player_name"]).drop_duplicates(subset=["player_id"])

# ---------------------------------------------------------
# 2. NAME NORMALIZATION (Crucial for Exact Matching)
# ---------------------------------------------------------
def normalize_name(name):
    """Lowercases and strips accents (e.g., CREPEAU -> crepeau)"""
    if pd.isna(name): return ""
    name = str(name).lower().strip()
    return ''.join(c for c in unicodedata.normalize('NFKD', name) if not unicodedata.combining(c))

df_squads['name_norm'] = df_squads['name'].apply(normalize_name)
df_master['master_name_norm'] = df_master['player_name'].apply(normalize_name)

# ---------------------------------------------------------
# 3. STEP 1: EXACT MATCHING (FIXED)
# ---------------------------------------------------------
print("Executing Exact Name Match...")

# Create a safe 1:1 dictionary, dropping duplicate names to prevent explosion
master_exact_dict = df_master.drop_duplicates(subset=['master_name_norm']).set_index('master_name_norm')['player_id'].to_dict()

# Map the IDs directly (bypasses alignment and duplication issues)
df_squads['transfermarkt_player_id'] = df_squads['name_norm'].map(master_exact_dict)

# Tag the match type
df_squads['match_type'] = None  # Initialize column
df_squads.loc[df_squads['transfermarkt_player_id'].notna(), 'match_type'] = 'Exact'

# ---------------------------------------------------------
# 4. STEP 2: FUZZY MATCHING (For the unresolved)
# ---------------------------------------------------------
unresolved = df_squads[df_squads['transfermarkt_player_id'].isna()].copy()
print(f"Executing Fuzzy Match for {len(unresolved)} unresolved players...")

master_name_dict = dict(zip(df_master['player_id'], df_master['master_name_norm']))
fuzzy_threshold = 85 # Adjust stringency as needed

for idx, row in unresolved.iterrows():
    search_name = row['name_norm']
    best_match = process.extractOne(search_name, master_name_dict, scorer=fuzz.token_sort_ratio)
    
    if best_match and best_match[1] >= fuzzy_threshold:
        # Extract the key (ID) for the matched value
        matched_tm_id = list(master_name_dict.keys())[list(master_name_dict.values()).index(best_match[0])]
        
        df_squads.at[idx, 'transfermarkt_player_id'] = matched_tm_id
        df_squads.at[idx, 'match_type'] = f'Fuzzy ({best_match[1]}%)'

# ---------------------------------------------------------
# 5. COVERAGE AUDIT
# ---------------------------------------------------------
total_squad_players = len(df_squads)
total_matched = df_squads['transfermarkt_player_id'].notna().sum()
exact_count = (df_squads['match_type'] == 'Exact').sum()
fuzzy_count = df_squads['match_type'].str.contains('Fuzzy', na=False).sum()
total_unresolved = total_squad_players - total_matched

print("\n" + "=" * 60)
print("📊 STEP 3: COVERAGE AUDIT")
print("=" * 60)
print(f"Total Squad Players:   {total_squad_players:,}")
print(f"Successfully Matched:  {total_matched:,} / {total_squad_players:,} ({(total_matched/total_squad_players)*100:.1f}%)")
print(f"  ↳ Exact Matches:     {exact_count:,}")
print(f"  ↳ Fuzzy Matches:     {fuzzy_count:,}")
print(f"Unresolved (Manual):   {total_unresolved:,}")

# ---------------------------------------------------------
# 6. OUTPUT EXTRACTION
# ---------------------------------------------------------
# Create the isolated mapping table
mapping_table = df_squads[['wc_player_id', 'name', 'transfermarkt_player_id', 'match_type']].copy()

print("\n--- Unresolved Players (Requires Manual Mapping) ---")
display(mapping_table[mapping_table['transfermarkt_player_id'].isna()].head(10))

# Save the mapping table for manual review/corrections
mapping_table.to_csv("../data/processed/squad_to_tm_mapping.csv", index=False)
print("\n✅ Mapping table saved to: ../data/processed/squad_to_tm_mapping.csv")
print("=" * 60)

🔗 WAREHOUSE MAPPING: SQUADS TO TRANSFERMARKT
Executing Exact Name Match...
Executing Fuzzy Match for 386 unresolved players...

📊 STEP 3: COVERAGE AUDIT
Total Squad Players:   1,248
Successfully Matched:  929 / 1,248 (74.4%)
  ↳ Exact Matches:     862
  ↳ Fuzzy Matches:     67
Unresolved (Manual):   319

--- Unresolved Players (Requires Manual Mapping) ---


,wc_player_id,name,transfermarkt_player_id,match_type
0,0,Dayne ST. CLAIR,NaN,None
1,1,Maxime CREPEAU,NaN,None
2,2,Owen GOODMAN,NaN,None
6,6,Joel WATERMAN,NaN,None
10,10,Richie LARYEA,NaN,None
12,12,Mathieu CHOINIERE,NaN,None
16,16,Jacob SHAFFELBURG,NaN,None
18,18,Nathan SALIBA,NaN,None
24,24,Ali AHMED,NaN,None
26,26,Raul RANGEL,NaN,None



✅ Mapping table saved to: ../data/processed/squad_to_tm_mapping.csv


In [23]:
print("=" * 60)
print("🌍 SQUAD COVERAGE REPORT BY COUNTRY")
print("=" * 60)

# We group by 'country' (or 'team_slug' if 'country' isn't clean)
# using the df_squads dataframe from our mapping step
coverage_data = []

for team, group in df_squads.groupby('country'):
    squad_size = len(group)
    matched = group['transfermarkt_player_id'].notna().sum()
    unresolved = squad_size - matched
    coverage_pct = (matched / squad_size) * 100
    
    coverage_data.append({
        'Team': team,
        'Squad Size': squad_size,
        'Matched': matched,
        'Unresolved': unresolved,
        'Coverage': coverage_pct
    })

# Convert to DataFrame
df_coverage = pd.DataFrame(coverage_data)

# Sort by Coverage (Lowest first to identify the problem areas)
df_coverage = df_coverage.sort_values(by=['Coverage', 'Team'], ascending=[True, True]).reset_index(drop=True)

# Create a display copy to cleanly format the percentages without altering the underlying numeric data
df_coverage_display = df_coverage.copy()
df_coverage_display['Coverage'] = df_coverage_display['Coverage'].map(lambda x: f"{x:.1f}%")

print("\n--- Full Coverage Audit (Worst to Best) ---")
display(df_coverage_display)

# Isolate the critical problem areas
problem_teams = df_coverage[df_coverage['Coverage'] < 50]
if not problem_teams.empty:
    print("\n🚨 CRITICAL ALERT: Teams with < 50% Coverage")
    print("These nations are driving the 319 unresolved count and will rely heavily on cold-start imputation.")
    display(problem_teams[['Team', 'Squad Size', 'Matched', 'Coverage']].copy().assign(
        Coverage=problem_teams['Coverage'].map(lambda x: f"{x:.1f}%")
    ))

print("\n" + "=" * 60)

🌍 SQUAD COVERAGE REPORT BY COUNTRY

--- Full Coverage Audit (Worst to Best) ---


,Team,Squad Size,Matched,Unresolved,Coverage
0,Jordan,26,2,24,7.7%
1,South Africa,26,2,24,7.7%
2,Korea Republic,26,5,21,19.2%
3,Qatar,26,5,21,19.2%
4,Saudi Arabia,26,8,18,30.8%
5,Panama,26,9,17,34.6%
6,Uzbekistan,26,10,16,38.5%
7,Egypt,26,12,14,46.2%
8,Iraq,26,13,13,50.0%
9,IR Iran,26,14,12,53.8%



🚨 CRITICAL ALERT: Teams with < 50% Coverage
These nations are driving the 319 unresolved count and will rely heavily on cold-start imputation.


,Team,Squad Size,Matched,Coverage
0,Jordan,26,2,7.7%
1,South Africa,26,2,7.7%
2,Korea Republic,26,5,19.2%
3,Qatar,26,5,19.2%
4,Saudi Arabia,26,8,30.8%
5,Panama,26,9,34.6%
6,Uzbekistan,26,10,38.5%
7,Egypt,26,12,46.2%
